<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook03_Erasing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Drive mount without torch to avoid conflicts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

ESD installation following the readme.md (Gondikota et al. 2023)
It will conflict

In [ ]:
!git clone https://github.com/rohitgandikota/erasing.git /content/erasing
%cd /content/erasing
#conda lines removed because running on colab instead of local environment
!pip install -r requirements.txt -q

!pip freeze>/content/drive/MyDrive/MyDissertationCN6000/requirements_esd.txt
print("ESD installed")
print("Restart session, rerun cell 1 and skip to cell 3")

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/MyDissertationCN6000'
os.chdir('/content/erasing') #changed direction from root to esd specific

# Silence HF download progress bars to prevent the GitHub-render widget bug
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# bfloat16 (hardcoded in the script) requires Ampere or newer
assert any(x in gpu_name for x in ["A100", "L4", "H100", "A10"]), (
    f"GPU {gpu_name} may not support bfloat16. Need A100/L4/H100/A10."
)

# Confirm the ESD env is in effect
import diffusers, transformers
print(f"diffusers: {diffusers.__version__}  (expect 0.37.x)")
print(f"transformers: {transformers.__version__}  (expect 5.x)")
print(f"torch: {torch.__version__}  (expect 2.11.x)")

In [ ]:
!pip uninstall torch_xla -y #to run before esd training

Running ESD training

In [ ]:
import datetime
date_str = datetime.date.today().isoformat()
log_path = f"/content/drive/MyDrive/MyDissertationCN6000/logs/training_log_{date_str}.txt"

!mkdir -p /content/drive/MyDrive/MyDissertationCN6000/logs

!python esd_sd.py \
    --erase_concept "Vincent van Gogh" \
    --train_method "esd-x" \
    --iterations 1000 \
    --negative_guidance 2 2>&1 | tee "{log_path}"

In [ ]:
import shutil
import datetime
from pathlib import Path

date_str = datetime.date.today().isoformat()

search_dirs = [Path("/content/erasing/esd-models/sd"), Path("/content/erasing")]
candidates = []
for d in search_dirs:
    if d.exists():
        candidates.extend(d.rglob("*.safetensors"))
        candidates.extend(d.rglob("*.pt"))

assert candidates, "No checkpoint found. Inspect /content/erasing/ manually."
candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
source = candidates[0]
print(f"Found checkpoint: {source}")
print(f"Size: {source.stat().st_size / 1e9:.2f} GB")

target = Path(f"/content/drive/MyDrive/MyDissertationCN6000/checkpoints/esd_vangogh_{date_str}{source.suffix}")
target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(source, target)
print(f"Copied to: {target}")